### Procesamiento de Lenguaje Natural I

# **Desafío 1**

### Alejandro Valle


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn


In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos


Cargamos los datos (ya separados de forma predeterminada en train y test)


El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).


In [ ]:
newsgroups_train = fetch_20newsgroups(
    subset="train", remove=("headers", "footers", "quotes")
)
newsgroups_test = fetch_20newsgroups(
    subset="test", remove=("headers", "footers", "quotes")
)

## Vectorización


Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html


In [ ]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto


In [ ]:
print(newsgroups_train.data[0])

Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.


In [ ]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.


In [ ]:
print(type(X_train))
print(f"shape: {X_train.shape}")
print(f"Cantidad de documentos: {X_train.shape[0]}")
print(f"Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}")

Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.


In [ ]:
tfidfvect.vocabulary_["car"]

Probamos con una palbra que no está en el documento.


In [ ]:
tfidfvect.vocabulary_["cocoliso"]

Es muy útil tener el diccionario opuesto que va de índices a términos


In [ ]:
idx2word = {v: k for k, v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros


In [ ]:
y_train = newsgroups_train.target
y_train[:10]

Hay 20 clases correspondientes a los 20 grupos de noticias


In [ ]:
print(f"clases {np.unique(newsgroups_test.target)}")
newsgroups_test.target_names

## Similaridad de documentos


Veamos similaridad de documentos. Tomemos algún documento


In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

Medimos la similaridad coseno con todos los documentos de train


In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor


In [ ]:
np.sort(cossim)[::-1]

Después vemos a qué documentos corresponden


In [ ]:
np.argsort(cossim)[::-1]

Obtenemos los 5 documentos más similares:


In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

El documento original pertenece a la clase:


In [ ]:
newsgroups_train.target_names[y_train[idx]]

Revisamos las clases de los 5 más similares:


In [ ]:
for i in mostsim:
    print(newsgroups_train.target_names[y_train[i]])

### Modelo de clasificación Naïve Bayes


Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn


In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.


In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred = clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

- El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
- El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.


In [ ]:
f1_score(y_test, y_pred, average="macro")

---


## **Consigna del Desafío 1**

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**


**1. Vectorizar documentos**

- Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
  Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
  la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**

- Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

- F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
  de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**

- De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
- Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


**1. Vectorizar documentos**

- Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
  Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
  la similaridad según el contenido del texto y la etiqueta de clasificación.


In [ ]:
import textwrap

rng = np.random.default_rng(42)
indexes = rng.choice(X_train.shape[0], size=5, replace=False)

N_CHARS = 600
WRAP_WIDTH = 90


def preview(text, n=N_CHARS, indent="     "):
    clean = " ".join(text.split())
    if not clean:
        return indent + "(documento vacío tras remover headers/footers/quotes)"
    clean = clean[:n] + ("..." if len(clean) > n else "")
    return textwrap.fill(
        clean, width=WRAP_WIDTH, initial_indent=indent, subsequent_indent=indent
    )


for index in indexes:
    cos_sim = cosine_similarity(X_train[index], X_train)[0]
    most_sim = np.argsort(cos_sim)[::-1][1:6]

    print(f"\n{'='*80}")
    print(
        f"Documento #{index}  |  Clase: {newsgroups_train.target_names[y_train[index]]}"
    )
    print(f"{'-'*80}")
    print(preview(newsgroups_train.data[index], indent=""), "\n")

    print("Documentos más similares:")
    for rank, doc in enumerate(most_sim, start=1):
        clase = newsgroups_train.target_names[y_train[doc]]
        print(f"  {rank}. Doc #{doc:<6} cos={cos_sim[doc]:.4f}  clase={clase}")
        print(preview(newsgroups_train.data[doc]), "\n")

    print()

**Comentarios:**

Similitud coseno con TF-IDF parece funcionar adecuadamente en relacionar contexto de los documentos. Las clases aletorias de los documentos elegidos se correlacionan correctamente (en algunos casos) con documentos de la misma clase (aunque no necesariamente en contenido). Por ejemplo, el documento de la clase `comp.sys.mac.hardware` tuvo 2 de sus 5 vecinos totales de la misma clase, pero las clases restantes de diferente categoría son de la misma temática (puertos de impresoras).

La similaridad agrupa por vocabulario compartido, por lo tanto, es una buena aproximación léxica aunque no capture contexto semántico.


**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**

- Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

knn = KNeighborsClassifier(n_neighbors=1, metric="cosine")
knn.fit(X_train, y_train)

knn_pred = knn.predict(X_test)

print(knn_pred)

f1_score(y_test, knn_pred, average="macro")

**Comentarios:**

Este es un enfoque sin entrenamiento `zero-shot` por que no se ajustan parámetros a partir de los datos. La consigna pide encontrar el documento con mayor similaridad, KNN es una buena aplicación para este problema con su métrica `cosine`. Se obtuvo un F1-Score menor que el NB de arriba. Además, es computacionalmente más costoso en inferencia: no hay entrenamiento pero cada predicción requiere comparar contra los 11314 documentos de entrenamiento mientras que NB solo compara contra 20 vectores de clase.


**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

- F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
  de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

  **NO cambiar el hiperparámetro ngram_range de los vectorizadores**.


**Baseline del NB original del notebook 0.5854345727938506**

**Baseline del zero-shot model 0.5049911553681621**

Estos son los valores a superar.


In [ ]:
# Balance de las clases

import pandas as pd


def class_balance(y, target_names, name):
    counts = pd.Series(y).value_counts().sort_index()
    df = pd.DataFrame(
        {
            "clase": [target_names[i] for i in counts.index],
            "n_docs": counts.values,
            "proporcion": (counts.values / counts.sum()).round(4),
        }
    )
    df.insert(0, "set", name)
    return df


balance = pd.concat(
    [
        class_balance(y_train, newsgroups_train.target_names, "train"),
        class_balance(y_test, newsgroups_test.target_names, "test"),
    ]
)

balance.pivot(index="clase", columns="set", values=["n_docs", "proporcion"])

ComplementNB suele usarse para datasets desbalaneados. No tenemos un dataset altamente desbalanceado, por lo que no se realizan tecnicas de balanceo.


Instancio un vectorizador con algunos parametros adicionales como experimentación. Estos documentos son textos de forums en el que muchas temáticas son similares entre sí, además que es muy probable que se use lenguaje coloquial. Hace sentido ignorar palabras raras y/o palabras que aparecen en todos los documentos.


In [ ]:
tfidf_v2 = TfidfVectorizer(
    min_df=5,  # ignora palabras que aparecen en menos de 5 documentos
    max_df=0.9,  # ignora palabras que aparecen en más del 90% de los documentos
    stop_words="english",
    sublinear_tf=True,  # usa 1 + log(tf) en vez de tf lineal. Ayuda a amortiguar el efecto de palabras que se repiten muchas veces en un mismo documento
)

X_train_v2 = tfidf_v2.fit_transform(newsgroups_train.data)
X_test_v2 = tfidf_v2.transform(newsgroups_test.data)

print(f"Vocabulario nuevo: {X_train_v2.shape[1]} términos")
print(f"Vocabulario original: {X_train.shape[1]} términos")

Pruebo con distintos valores de suavizado. Si una palabra nunca aparece en los docs de entrenamiento de una clase, la probabilidad estimada sería 0, lo cual rompe el cálculo para cualqueir documento de test que si contenga esa palabra. El valor por defecto es 1, es como "simular" de que si ha visto una palabra inexistente ya que un alpha 0 pues anula todo el producto.

Se experimentará con este hiperparámetro.


In [ ]:
alphas = [0.001, 0.01, 0.1, 0.5, 1, 5]
models = {"MultinomialNB": MultinomialNB, "ComplementNB": ComplementNB}

results = []

for model_name, model_class in models.items():
    for alpha in alphas:
        clf = model_class(alpha=alpha)
        clf.fit(X_train_v2, y_train)
        y_pred = clf.predict(X_test_v2)
        f1 = f1_score(y_test, y_pred, average="macro")
        results.append({"modelo": model_name, "alpha": alpha, "f1_macro": f1})
        print(f"{model_name:15s} alpha={alpha:<6} F1-macro={f1:.4f}")

In [ ]:
results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
results_df

best = results_df.iloc[0]
print(
    f"Mejor modelo: {best['modelo']} (alpha={best['alpha']}) con F1-macro={best['f1_macro']:.4f}"
)

In [ ]:
# Aislamos el efecto del vectorizador nuevo del efecto del tuneo de alpha/modelo
# usando alpha=1 (default) como referencia común en ambos vectorizadores

clf_baseline = MultinomialNB()  # alpha=1 default
clf_baseline.fit(X_train, y_train)
f1_baseline = f1_score(y_test, clf_baseline.predict(X_test), average="macro")

f1_vectorizer_only = results_df.loc[
    (results_df["modelo"] == "MultinomialNB") & (results_df["alpha"] == 1), "f1_macro"
].iloc[0]

f1_full_tuning = results_df.iloc[0]["f1_macro"]

print(
    f"Baseline (vectorizador original, alpha=1 default):      F1-macro = {f1_baseline:.4f}"
)
print(
    f"Solo vectorizador nuevo (alpha=1 default):               F1-macro = {f1_vectorizer_only:.4f}"
)
print(
    f"Vectorizador nuevo + mejor alpha/modelo ({best['modelo']}, alpha={best['alpha']}): "
    f"F1-macro = {f1_full_tuning:.4f}"
)
print()
print(
    f"Mejora atribuible al vectorizador:          {f1_vectorizer_only - f1_baseline:+.4f}"
)
print(
    f"Mejora atribuible al tuneo de alpha/modelo: {f1_full_tuning - f1_vectorizer_only:+.4f}"
)

**Comentarios:**

- Vectorizer: Agregando unos params mas respecto a la configuración default hecha más arriba se redujo el vocabulario por aproximadamente un 82%. De 101631 a 17797. A pesar de tener muchisima menos dimensionalidad, el F1-macro mejoró respecto al baseline original hasta un máximo de 0.682. Esto sugiere que gran parte de ese vocabulario reducido eran ruido (términos raros, comunes, repetitivos, etc)
- Alpha: el comportamiento de alpha en MultiNomialNb no es monótono. Se llega a un pico en el valor default. Por el contrario, ComplementNB parece menos sensible al hiperpárametro `alpha` ya que la variación entre el peor aplha y el mejor es mucho menor que en MultiNomialNB. En MultinomialNB utiliza conteos por clase individual el cual puede ser pequeño, en ComplementNB se utiliza los conteos agregados de casi todo el corpus, el cual puede ser mucho más grande (Estos conteos estan en el denominador de la ecuación junto con el alpha).
- Conclusiones: Con la experimentación en la anterior celda se le atribuye la mejora en F1-score a ambos ajustes hechos: el ajuste de parametros del vectorizador y ComplementNB
